# Notebook 04 — CLIP-B/32 Pipeline
**Owner: Person A — Week 2**

Full CLIP-B/32 pipeline: model load, SAE verification, 5,000-image activation cache,
top-activating patches, CLIP auto-labeling, and Monosemanticity Scores for layers 4, 6, 9.

Mirrors notebook 01 + 02 for DINO but uses `configs/clip_b32.yaml`.
Outputs feed notebook 03 (CaFE locality analysis) for the CLIP run.

**Key differences vs. DINO:**
- 50 tokens per image (1 CLS + 49 patches, 7×7 grid at patch_size=32)
- CLIP normalisation (not ImageNet) — handled automatically by `preprocess_image`
- SAE checkpoints from `Prisma-Multimodal/sparse-autoencoder-clip-b-32-*`
- Cache written to `outputs/caches/activations_clip.h5`

In [ ]:
"""Environment — run this cell first.

Reloads config to clip_b32.yaml so every src/ import reads CLIP settings
(patch_size=32, num_registers=0, CLIP SAE repo IDs, etc.).
Do NOT call get_config() without reload=True in subsequent cells — the module-level
cache will already be warm with the CLIP config after this cell.
"""
import gzip, pickle, json, math
from pathlib import Path

import torch
from src.config import get_config
import src.config as _cfg_mod

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"          # use GPU device 2; PyTorch sees it as cuda:0
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduces fragmentation


cfg       = get_config("../configs/clip_b32.yaml", reload=True)
repo_root = _cfg_mod._DEFAULT_CONFIG.parent.parent

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Config : {cfg.model.name}")
print(f"Device : {device}")
print(f"Layers : {cfg.sae.target_layers}")
if device == "cuda":
    free  = torch.cuda.mem_get_info()[0] / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU    : {torch.cuda.get_device_name(0)}  {free:.1f}/{total:.1f} GB free")

## 1. Load CLIP model and forward-pass smoke test

In [ ]:
"""Step 1 — load CLIP-B/32 via Prisma HookedViT and run a smoke test.

Expected output:
  - activations.shape: (1, 50, 768)   — 1 CLS + 49 patch tokens, d_model=768
  - hook key format:   blocks.{N}.hook_resid_post  (same as DINO)
"""
from src.model import get_model, preprocess_image

model = get_model()   # loads openai/clip-vit-base-patch32 with is_clip=True

# Find any .jpg in the imagenet_val directory
data_dir   = repo_root / cfg.data.imagenet_val_path
image_path = next(data_dir.glob("**/*.jpg"), None) or next(data_dir.glob("**/*.JPEG"), None)
assert image_path is not None, f"No images found in {data_dir}"

pixels = preprocess_image(str(image_path)).to(device)
print(f"Image  : {image_path.name}  pixels shape: {tuple(pixels.shape)}")

with torch.no_grad():
    logits, act_cache = model.run_with_cache(pixels)

hook_key = f"blocks.{cfg.sae.primary_layer}.hook_resid_post"
assert hook_key in act_cache, f"Expected key '{hook_key}' not in cache"
activations = act_cache[hook_key]
print(f"activations.shape: {activations.shape}")  # expect (1, 50, 768)

seq_len = activations.shape[1]
expected_seq = 1 + (cfg.model.image_size // cfg.model.patch_size) ** 2
assert seq_len == expected_seq, (
    f"seq_len={seq_len} but expected {expected_seq} "
    f"(1 CLS + {expected_seq-1} patches @ patch_size={cfg.model.patch_size})"
)
print(f"Sequence length OK: {seq_len} tokens "
      f"(1 CLS + {seq_len-1} patches in {int(seq_len-1)**0.5:.0f}×{int(seq_len-1)**0.5:.0f} grid)")

# Cache check
assert get_model() is model, "get_model() caching broken"
print("Model cache check passed.")

## 2. Download and verify CLIP SAEs

Run the download script once (skip if weights already present):
```
python utils/download_saes.py --config configs/clip_b32.yaml
```
Saves to `outputs/saes/clip-vit-base-patch32/layer_{4,6,9}/`

In [ ]:
"""Step 2 — verify CLIP SAEs load and pass reconstruction / L0 checks."""
from src.sae import get_sae, encode, decode, get_l0_sparsity, get_reconstruction_loss

for layer in cfg.sae.target_layers:
    sae = get_sae(layer)    # model_id defaults to cfg.model.name (clip-vit-base-patch32)
    print(f"\nLayer {layer}")
    print(f"  d_in={sae.d_in}  d_sae={sae.d_sae}")
    print(f"  W_enc: {sae.W_enc.shape}  W_dec: {sae.W_dec.shape}")

    acts = act_cache[f"blocks.{layer}.hook_resid_post"]  # (1, 50, 768) from cell 1
    feats = encode(acts, layer)                           # (1, 50, d_sae)
    recon = decode(feats, layer)                          # (1, 50, 768)
    print(f"  features shape: {tuple(feats.shape)}")

    l0   = get_l0_sparsity(acts, layer)
    loss = get_reconstruction_loss(acts, layer)
    print(f"  L0={l0:.1f}  (target < {cfg.sae.l0_target})")
    print(f"  Reconstruction loss={loss:.4f}  (target < 0.05)")

    if l0 >= cfg.sae.l0_target:
        print(f"  WARNING: L0 {l0:.1f} exceeds target {cfg.sae.l0_target}")
    if loss >= 0.05:
        print(f"  WARNING: reconstruction loss {loss:.4f} exceeds 0.05 target")

## 3. Build CLIP activation cache

Uses the same 5,000 ImageNet val images as the DINO cache.
Written to `outputs/caches/activations_clip.h5` (config: `cfg.outputs.cache_path`).
Estimated time: ~15–30 min on A5000 for 5,000 images × 3 layers.

In [ ]:
"""Step 3a — collect image paths (same flat dir as DINO, same 5000 images)."""
import zipfile

data_dir  = repo_root / cfg.data.imagenet_val_path
zip_path  = repo_root / "data" / "imagenet_val_5000.zip"
cache_path = repo_root / cfg.outputs.cache_path  # outputs/caches/activations_clip.h5

flat_dir = data_dir
if zip_path.exists() and not (flat_dir.exists() and any(flat_dir.glob("*.jpg"))):
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(repo_root / "data")

image_paths = sorted(str(p.resolve()) for p in flat_dir.glob("*.jpg"))
labels    = ["unknown"] * len(image_paths)
class_ids = [-1] * len(image_paths)

print(f"Images   : {len(image_paths)}")
print(f"Cache    : {cache_path}")
print(f"Layers   : {cfg.sae.target_layers}")

In [ ]:
"""Step 3b — build cache (run once, ~15-30 min on A5000)."""
from src.cache import build_cache

if cache_path.exists():
    print(f"Cache already exists at {cache_path} — skipping build.")
    print("Delete the file and re-run to rebuild.")
else:
    print(f"Building CLIP cache for {len(image_paths)} images, layers {cfg.sae.target_layers} ...")
    build_cache(image_paths, labels, class_ids, outputpath=str(cache_path))
    print(f"Done. Cache saved to {cache_path}")

In [ ]:
"""Step 3c — verify cache contents."""
from src.cache import load_metadata, load_image_index, load_layer

meta  = load_metadata(cachepath=str(cache_path))
index = load_image_index(cachepath=str(cache_path))

print(f"Model    : {meta['model_name']}")
print(f"Layers   : {meta['layers']}")
print(f"Images   : {meta['n_images']}")

expected_tokens = 1 + (cfg.model.image_size // cfg.model.patch_size) ** 2  # 50 for CLIP-B/32
for layer in cfg.sae.target_layers:
    acts = load_layer(layer, cachepath=str(cache_path))
    print(f"  layer {layer}: {tuple(acts.shape)}")  # expect (5000, 50, 768)
    assert acts.shape[1] == expected_tokens, (
        f"layer {layer}: expected seq_len={expected_tokens}, got {acts.shape[1]}"
    )
print("Cache verification passed.")

## 4. Top patches — all features, all layers

Peak GPU memory: `batch_size × 50 × 32768 × 4 bytes`  
- batch_size=64 → ~0.4 GB (safe for any GPU)  
- batch_size=256 → ~1.6 GB (recommended on A5000)

Results saved to `outputs/features/top_patches_clip_layer{N}_full.pkl.gz`.

In [ ]:
"""Step 4 — top-k patches for all SAE features, streamed from cache."""
from src.features import get_top_patches_all_features_from_cache

feature_out_dir = repo_root / "outputs" / "features"
feature_out_dir.mkdir(parents=True, exist_ok=True)

all_top_patches_by_layer = {}

for layer in cfg.sae.target_layers:
    pkl_path = feature_out_dir / f"top_patches_clip_layer{layer}_full.pkl.gz"

    if pkl_path.exists():
        with gzip.open(pkl_path, "rb") as f:
            all_top_patches_by_layer[layer] = pickle.load(f)
        print(f"[cached] layer {layer}: {len(all_top_patches_by_layer[layer]):,} features")
    else:
        print(f"Computing top patches for layer {layer} ...")
        patches = get_top_patches_all_features_from_cache(
            layer,
            index["paths"],
            cachepath=str(cache_path),
            batch_size=256,
        )
        with gzip.open(pkl_path, "wb") as f:
            pickle.dump(patches, f, protocol=pickle.HIGHEST_PROTOCOL)
        all_top_patches_by_layer[layer] = patches
        print(f"Saved layer {layer}: {len(patches):,} features → {pkl_path}")

## 5. CLIP auto-labeling and Monosemanticity Scores

Same flow as notebook 02 (DINO) — two steps:
- **5a** Precompute patch embeddings (once per session)
- **5b** CLIP label all features
- **5c** Compute and save MS scores, plot distribution

In [ ]:
"""Step 5a — load CLIP labeler and precompute patch embeddings (in-memory only)."""
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"  # use cached pytorch_model.bin; avoids 605 MB XetHub re-download of model.safetensors

from src.features import load_clip_labeler, precompute_patch_embeddings
from utils.clip_vocab import get_vocab

clip_model, processor = load_clip_labeler()
vocab = get_vocab()
print(f"Vocab: {len(vocab):,} concepts")

patch_embeddings_by_layer = {}
for layer in cfg.sae.target_layers:
    print(f"Precomputing patch embeddings for layer {layer} ...")
    patch_embeddings_by_layer[layer] = precompute_patch_embeddings(
        all_top_patches_by_layer[layer], clip_model, processor, batch_size=256
    )

In [ ]:
"""Step 5b — CLIP label all features, then remove dead features.

top_n=10 so tier_promoted_label in step 6 can promote a Tier 1/2 descriptor
from anywhere in the ranked list. Mirrors notebook 02 cell c04.

Dead features (max activation <= cfg.sae.dead_feature_threshold) produce
spurious identical labels (nearest vocab word to a zero embedding). We label
everything then drop those entries so downstream cells never see them.
"""
from src.features import label_features_clip

dead_threshold = cfg.sae.dead_feature_threshold

feature_labels_by_layer = {}
for layer in cfg.sae.target_layers:
    labels_cache = feature_out_dir / f"clip_labels_clip_layer{layer}_full.json"
    if labels_cache.exists():
        with open(labels_cache, encoding="utf-8") as f:
            feature_labels_by_layer[layer] = {int(k): v for k, v in json.load(f).items()}
        print(f"[cached] layer {layer}: {len(feature_labels_by_layer[layer]):,} labels")
    else:
        labels = label_features_clip(
            all_top_patches_by_layer[layer], vocab, clip_model, processor,
            top_n=10, patch_embeddings=patch_embeddings_by_layer[layer],
        )
        with open(labels_cache, "w", encoding="utf-8") as f:
            json.dump({str(k): v for k, v in labels.items()}, f, indent=2)
        feature_labels_by_layer[layer] = labels
        print(f"Saved layer {layer}: {len(labels):,} labels → {labels_cache}")

    # Remove dead features: max_act <= dead_threshold produces spurious labels
    patches = all_top_patches_by_layer[layer]
    before = len(feature_labels_by_layer[layer])
    feature_labels_by_layer[layer] = {
        fi: lbls
        for fi, lbls in feature_labels_by_layer[layer].items()
        if max((p["activation_value"] for p in patches.get(fi, [])), default=0.0) > dead_threshold
    }
    dead = before - len(feature_labels_by_layer[layer])
    print(f"  Removed {dead:,} dead features (max_act <= {dead_threshold}) — {len(feature_labels_by_layer[layer]):,} alive")

    sample = list(feature_labels_by_layer[layer].items())[:3]
    for fi, lbls in sample:
        print(f"  feat {fi}: {lbls}")

In [ ]:
"""Step 5c — compute MS scores and plot distribution per layer."""
from src.features import compute_monosemanticity_score
from src.visualise import plot_monosemanticity_distribution

figures_dir = repo_root / cfg.outputs.report_figures_path
figures_dir.mkdir(parents=True, exist_ok=True)

MS_MAX_PATCHES = cfg.features.ms_max_patches
ms_scores_by_layer = {}

for layer in cfg.sae.target_layers:
    scores_path = feature_out_dir / f"ms_scores_clip_layer{layer}_top{MS_MAX_PATCHES}.json"
    if scores_path.exists():
        with open(scores_path, encoding="utf-8") as f:
            ms_scores_by_layer[layer] = {int(k): v for k, v in json.load(f).items()}
        print(f"[cached] layer {layer}: {len(ms_scores_by_layer[layer]):,} MS scores")
    else:
        scores = compute_monosemanticity_score(
            all_top_patches_by_layer[layer],
            patch_embeddings_by_layer[layer],
            max_patches=MS_MAX_PATCHES,
        )
        with open(scores_path, "w", encoding="utf-8") as f:
            json.dump({str(k): v for k, v in scores.items()}, f)
        ms_scores_by_layer[layer] = scores
        print(f"Saved layer {layer}: {len(scores):,} MS scores → {scores_path}")

    ms_scores = ms_scores_by_layer[layer]
    alive = [v for v in ms_scores.values() if v is not None and not math.isnan(v)]
    dead  = sum(1 for v in ms_scores.values() if v is None or math.isnan(v))
    print(f"  Layer {layer}: {len(alive):,} active, {dead:,} dead")
    print(f"  Median MS: {sorted(alive)[len(alive)//2]:.3f}")
    print(f"  80th pct : {sorted(alive)[int(len(alive)*0.8)]:.3f}")

    fig = plot_monosemanticity_distribution(
        ms_scores,
        layer=layer,
        save_path=str(figures_dir / f"ms_dist_clip_layer{layer}_top{MS_MAX_PATCHES}.png"),
    )
    display(fig)

## 6. Feature gallery — top 50 by MS score

In [ ]:
"""Step 6 — feature gallery — top 50 by MS score with tier promotion and label de-duplication.

Mirrors notebook 02 cells 64f84ff5 + c06 combined.

Two additions vs the naive top-k:

1. Tier promotion (from nb02): walks each feature's top-10 label list and
   promotes the first Tier 1/2 hit (descriptive texture/part) over plain
   ImageNet class names. Falls back to top-1 (Tier 4) when none found.

2. Label de-duplication (new): after ranking by MS score, skips a feature
   whose promoted label already appears in the selected set. This prevents
   one dominant visual concept (e.g. piano keyboards) from filling the gallery.
   Each unique label is admitted at most once.
"""
from src.visualise import plot_feature_gallery
from utils.clip_vocab import TIER1, TIER2

_TIER1_SET = {s.lower() for s in TIER1}
_TIER2_SET = {s.lower() for s in TIER2}

def _tier_promoted_label(labels: list[str]) -> tuple[str, str]:
    for label in labels:
        if label.lower() in _TIER1_SET:
            return label, "T1"
        if label.lower() in _TIER2_SET:
            return label, "T2"
    return (labels[0], "T4") if labels else ("?", "T4")

layer = cfg.sae.primary_layer
ms_scores       = ms_scores_by_layer[layer]
all_top_patches = all_top_patches_by_layer[layer]
feature_labels  = feature_labels_by_layer[layer]

MIN_ACTIVATION = cfg.features.min_activation_threshold
MIN_PATCHES    = cfg.features.ms_max_patches
TOP_N          = cfg.features.monosemanticity_top_n

# Build promoted labels for every eligible feature
promoted_labels: dict[int, list[str]] = {}
tier_sources: dict[int, str] = {}
for fi, lbls in feature_labels.items():
    if not lbls:
        promoted_labels[fi] = []
        tier_sources[fi] = "?"
        continue
    best, tier = _tier_promoted_label(lbls)
    tier_sources[fi] = tier
    raw_top2 = [l for l in lbls[:3] if l != best][:2]
    promoted_labels[fi] = [best] + raw_top2

tier_counts = {}
for t in tier_sources.values():
    tier_counts[t] = tier_counts.get(t, 0) + 1
print("Tier promotion summary:", tier_counts)

# Support + activation filter (same as nb02)
eligible = []
for fi, score in ms_scores.items():
    if score is None or math.isnan(score):
        continue
    valid = [p for p in all_top_patches.get(fi, []) if p.get("patch_row") is not None]
    if len(valid) < MIN_PATCHES:
        continue
    if max((p["activation_value"] for p in valid), default=0.0) < MIN_ACTIVATION:
        continue
    eligible.append((fi, score))

eligible.sort(key=lambda kv: kv[1], reverse=True)

# De-duplicate by promoted label: keep the highest-MS representative per label
seen_labels: set[str] = set()
top_50: list[int] = []
skipped = 0
for fi, _ in eligible:
    if len(top_50) >= TOP_N:
        break
    top_label = (promoted_labels.get(fi) or ["?"])[0].lower()
    if top_label in seen_labels:
        skipped += 1
        continue
    seen_labels.add(top_label)
    top_50.append(fi)

print(f"Support filter (>= {MIN_PATCHES} patches, max_act >= {MIN_ACTIVATION}): "
      f"{len(eligible):,} eligible")
print(f"After label de-duplication: {len(top_50)} selected, {skipped} duplicates skipped")
if top_50:
    print(f"MS range: {ms_scores[top_50[0]]:.3f} … {ms_scores[top_50[-1]]:.3f}")

_labels = {
    fi: promoted_labels.get(fi, []) + [f"[{tier_sources.get(fi, '?')}]"]
    for fi in top_50
}

fig = plot_feature_gallery(
    all_top_patches, top_50,
    labels=_labels,
    scores=ms_scores,
    max_patches=MIN_PATCHES,
    save_path=str(figures_dir / "feature_gallery_clip.png"),
)
display(fig)
print(f"Saved → {figures_dir / 'feature_gallery_clip.png'}")

## Checkpoint

**Person A**
- [ ] `get_model()` returns CLIP model, `activations.shape == (1, 50, 768)`
- [ ] All three SAEs load; L0 < `cfg.sae.l0_target`; reconstruction loss < 0.05
- [ ] `outputs/caches/activations_clip.h5` built and verified (5,000 × 50 × 768 per layer)
- [ ] `top_patches_clip_layer{4,6,9}_full.pkl.gz` saved
- [ ] `ms_scores_clip_layer{4,6,9}_top5.json` saved
- [ ] MS distribution plots saved to `report/figures/`
- [ ] Feature gallery (`feature_gallery_clip.png`) saved

**Hand-off to notebook 03 (CaFE)**  
Load `configs/clip_b32.yaml` with `reload=True` before running nb03 cells on CLIP outputs.